In [ ]:
import os
from pathlib import Path

project_root = Path().resolve().parent.parent
os.chdir(project_root)
print("Current working directory:", os.getcwd())

In [3]:
from langfuse import observe

from rich import print
from pydantic import Field
from pydantic import BaseModel
from typing import List

from app.agents.object.aggent_factory import agent_factory
from app.utils.agents.prompt_loader import load_prompt


class AnswerSchema(BaseModel):
    answer: str = Field(
        description="Give the final solution to the problem",
    )
    reasoning: List[str] = Field(
        description="Describe step by step how did you came up with the answer",
    )


@observe()
def answer_math_problem(math_problem: str):

    agent_id = "math_teacher_agent"
    extractor_webpages_prompt = load_prompt(agent_id)
    user_prompt = extractor_webpages_prompt["user_prompt"].format(
        input_data=math_problem,
    )
    try:
        agent = agent_factory.get_agent(
            agent_id=agent_id,
            output_schema=AnswerSchema,
        )
        response, _ = agent.run(user_prompt)
        return response

    except Exception as e:
        print(f"[bold red]Agent failed:[/bold red] {e}")


if __name__ == "__main__":
    math_problem = """
    A school is organizing a field trip. They rent buses that hold 48 students each.
    - There are 365 students going on the trip.
    - 8 teachers must also ride the buses.
    - Every bus must have at least one teacher on it.
    What is the minimum number of buses needed to transport everyone?
    """

    # Answer should be 8: https://chatgpt.com/s/t_69aae98e777881918159e00b2bcacc3c
    result = answer_math_problem(math_problem)

    print("\n[bold green]Response successful![/bold green]")
    print(f"\nResult: {result.answer}")
    for i, step in enumerate(result.reasoning):
        print(f"Step [{i}]: {step}")


Response successful!

Result: 8 buses

Step [0]: Total students = 365.

Step [1]: Total teachers = 8.

Step [2]: Total people = 365 + 8 = 373.

...

Step [7]: Number of students per bus: 48.

Step [8]: Number of buses needed for students: 365 / 48 ≈ 7.6, so at least 8 buses for students.

Step [9]: Total buses needed: 8 (for teachers) and enough to seat all students, which is 8 buses.

Step [10]: Check if 8 buses are enough for students: 8 buses * 48 students = 384 seats, which is enough for 365 
students.

Step [11]: Since 8 buses are enough for both students and teachers, the minimum number of buses needed is 8.